# NFDI4Objects KG — Poseidon aDNA sites on a map

This notebook queries the [NFDI4Objects Knowledge Graph](https://graph.nfdi4objects.net/)
for archaeogenetic **aDNA samples** from the
[Poseidon Community Archive](https://www.poseidon-adna.org/) — loaded
into the N4O KG as `collection/17` under the
[ArNO ontology](https://github.com/archaeonatural-cloud/archaeonatural-ontology)
(Straten, Strohm, Thiery & Renz 2025) — aggregates them by discovery
site, and plots the result on two interactive
[folium](https://python-visualization.github.io/folium/) maps. The
first map shows individual sites as country-coloured markers; the
second shows a **hex-binned sample-density grid** so concentrations
of heavily sampled regions are visible at a glance.

It is part of an Open Educational Resource series on knowledge
graphs and linked open data, and is the local Jupyter companion to
`n4okg-poseidon-sites-map-live.qmd`.

## Requirements

```bash
pip install SPARQLWrapper pandas folium branca
```


## About this notebook

### Why this dataset?

`collection/17` holds the Poseidon Community Archive (PCA), a large
open corpus of archaeogenetic genotype data with contextual and
bibliographic metadata (Schmid et al. 2024). For teaching purposes
it offers something most linked-data-for-archaeology examples do
not: a **natural-science dataset** mapped into a cultural-heritage
knowledge graph. Samples come with coordinates, countries, dating,
capture types, and genetic measurements — enough structure to make
meaningful aggregations visible without being overwhelming.

### What you'll learn

- how to traverse a multi-hop ArNO path
  (`aDNASample → DiscoverySite → Site → Place → Country`) in a
  single SPARQL query
- how the `GeoSPARQL` two-step indirection works on this endpoint
  (geometry is attached to the `DiscoverySite`, not to the `Sample`)
- how to aggregate by a coordinate tuple so that one named site with
  multiple samples collapses to a single visible marker
- how to build a hex-binned density grid by hand and render it as
  `folium.Polygon` — no extra plugin, no canvas resize quirks

### Data-context notes

- **Coordinates live on the `DiscoverySite`, not on the `Sample`.**
  The ArNO modelling keeps the sample focused on what *is the
  sample* (tissue, genetics, dating) and delegates *where it was
  found* to the discovery site. The query path reflects this:
  `?sample arno:foundAtDiscoverySite ?ds . ?ds geo:hasGeometry/geo:asWKT ?wkt`.
- **Aggregation is by `(site, country, wkt)`** rather than by site
  name alone. A single named site may in principle be represented
  by more than one coordinate in the underlying data; grouping by
  the geometry tuple avoids accidentally unifying them.
- **WKT literals** on this endpoint use `POINT(lon lat)` (uppercase,
  longitude first). The parser below handles both casings
  defensively and swaps the order for folium / Leaflet.

### Tooling notes

This local variant uses `SPARQLWrapper` for the query, `pandas` for
the DataFrame, and `folium` for the maps (with `folium.plugins.Fullscreen`
for a fullscreen control and a `branca.MacroElement` custom legend
for the hex density map). Setting a descriptive `User-Agent` is not
optional for a public SPARQL endpoint — the service may throttle or
block requests with a generic UA.


## Step 1 — Define the SPARQL query

The query walks the ArNO path
`aDNASample → foundAtDiscoverySite → atSite → atPlace → inCountry`
to reach each sample's country, and in parallel pulls the WKT
geometry from the discovery site via `geo:hasGeometry/geo:asWKT`.
We then `GROUP BY` site label, country label, and WKT so that one
row of the result is one *place on the map*, regardless of how
many samples it represents.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"
USER_AGENT = "OER-Poseidon-SitesMap-Notebook/1.0"

SPARQL_QUERY = """
PREFIX arno: <http://archaeonatural.cloud/ont/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX geosparql: <http://www.opengis.net/ont/geosparql#>
SELECT ?siteLabel ?countryLabel ?wkt (COUNT(DISTINCT ?sample) AS ?numSamples)
WHERE {
  GRAPH <https://graph.nfdi4objects.net/collection/17> {
    ?sample rdf:type arno:aDNASample ;
            arno:foundAtDiscoverySite ?discoverySite .
    ?discoverySite geosparql:hasGeometry ?geom .
    ?geom geosparql:asWKT ?wkt .
    OPTIONAL {
      ?discoverySite arno:atSite ?site .
      ?site rdfs:label ?siteLabel .
    }
    OPTIONAL {
      ?discoverySite arno:atSite ?site .
      ?site arno:atPlace ?place .
      ?place arno:inCountry ?country .
      ?country rdfs:label ?countryLabel .
    }
  }
}
GROUP BY ?siteLabel ?countryLabel ?wkt
ORDER BY DESC(?numSamples)
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
# POST is required here: the GRAPH clause with long IRIs exceeds most
# URL-length limits when the query is sent via GET.
sparql.setMethod("POST")
sparql.setQuery(SPARQL_QUERY)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
bindings = results["results"]["bindings"]
print(f"✓ Retrieved {len(bindings)} rows from the N4O KG")

## Step 2 — Load the data

The bindings from a SPARQL JSON result are flattened into a
DataFrame. The WKT literal is split into separate `latitude` and
`longitude` columns, and we drop rows whose coordinates failed to
parse — GeoSPARQL literals occasionally carry non-standard
wrappings that the regex does not match, and it is better to
surface that gracefully than to fail the whole notebook.


In [ ]:
import re
import pandas as pd

assert bindings, "⚠ Please run the query cell first."

# Case-insensitive WKT parser: handles both 'Point(...)' and 'POINT(...)'
# and tolerates an optional SRID prefix.
_WKT_POINT_RE = re.compile(
    r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)", re.IGNORECASE
)

def parse_wkt_point(wkt):
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)

rows = []
for r in bindings:
    lat, lon = parse_wkt_point(r.get("wkt", {}).get("value"))
    rows.append({
        "site":       r.get("siteLabel", {}).get("value", "(unnamed site)"),
        "country":    r.get("countryLabel", {}).get("value", "(unknown)"),
        "wkt":        r.get("wkt", {}).get("value"),
        "numSamples": int(r["numSamples"]["value"]),
        "latitude":   lat,
        "longitude":  lon,
    })

df = pd.DataFrame(rows).dropna(subset=["latitude", "longitude"])
assert not df.empty, "⚠ No records with coordinates returned — check query."
print(f"✓ {len(df)} discovery-site coordinates across "
      f"{df['country'].nunique()} countries "
      f"(total samples: {df['numSamples'].sum()}).")
df.head(10)

## Step 3a — Map with country-coloured markers

Each discovery-site record becomes a `folium.CircleMarker`, coloured
by its country. One `FeatureGroup` per country gives an
interactive toggle via the `LayerControl` in the top-right corner
— unchecking a country hides its markers. The layer-control labels
embed colour swatches so the panel also acts as the legend.

Four base-tile options (OpenStreetMap default, OSM Humanitarian,
Esri World Imagery, Esri World Terrain) are available from the
same control.


In [ ]:
import folium
from folium import FeatureGroup
from folium.plugins import Fullscreen

assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

# --- Country -> colour (same palette as the qmd for visual consistency)
palette = [
    "#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#469990", "#9A6324", "#800000",
    "#808000", "#000075", "#e6beff", "#fabed4", "#aaffc3",
    "#fffac8", "#dcbeff", "#ffd8b1", "#a9a9a9", "#ffe119",
]
countries = sorted(df["country"].unique())
country_colors = {c: palette[i % len(palette)] for i, c in enumerate(countries)}

centre = [df["latitude"].mean(), df["longitude"].mean()]
m = folium.Map(location=centre, zoom_start=3, tiles=None)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    "https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    name="OSM Humanitarian",
    attr="&copy; OpenStreetMap contributors, Tiles style by HOT",
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Imagery",
    attr="Tiles &copy; Esri",
    max_zoom=18,
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Terrain",
    attr="Tiles &copy; Esri — Source: USGS, Esri, TANA",
    max_zoom=13,
).add_to(m)

Fullscreen(position="topleft").add_to(m)

# One FeatureGroup per country; swatch-in-label doubles as legend.
country_layers = {}
for c in countries:
    swatch = (f'<span style="display:inline-block;width:10px;height:10px;'
              f'background:{country_colors[c]};border-radius:50%;'
              f'margin-right:6px;vertical-align:middle"></span>')
    country_layers[c] = FeatureGroup(name=swatch + c, show=True)

for _, row in df.iterrows():
    country = row["country"]
    colour = country_colors[country]
    popup_html = (
        f'<b>{row["site"]}</b><br>'
        f'Country: {country}<br>'
        f'aDNA samples: <b>{row["numSamples"]}</b>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6, color=colour, weight=1.2,
        fill=True, fill_color=colour, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=260),
    ).add_to(country_layers[country])

for layer in country_layers.values():
    layer.add_to(m)

folium.LayerControl(collapsed=True).add_to(m)

# Fit the bounds to the actual data so the map centres nicely
m.fit_bounds([[df["latitude"].min(), df["longitude"].min()],
              [df["latitude"].max(), df["longitude"].max()]],
             padding=(20, 20))
m

## Step 3b — Hex-binned sample density

The second map aggregates samples into a **hexagonal grid**, with
each cell coloured by the number of aDNA samples falling inside
it. A site with hundreds of samples contributes *hundreds* to its
cell, not one — so the heatmap reflects **sampling intensity**,
not just the presence of a dig.

We hand-roll the hex binning in Python and draw the polygons as
`folium.Polygon`. The hex size is ~1.0° longitude (roughly 80 km
at 45°N), large enough to aggregate multiple neighbouring sites
into each cell, and small enough to keep regional structure
visible. A `LogNorm`-style colour ramp is used because aDNA sample
counts per cell are heavily right-skewed.


In [ ]:
import math
from collections import Counter
from branca.element import MacroElement
from jinja2 import Template

assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

# Weight each site by its sample count.
weighted_points = [
    (float(row["longitude"]), float(row["latitude"]), int(row["numSamples"]))
    for _, row in df.iterrows()
]

HEX_SIZE_DEG = 1.0
phi0 = math.radians(df["latitude"].mean())
ky   = 1.0 / math.cos(phi0)

def point_to_hex(lon, lat, size):
    x, y = lon, lat * ky
    q = (2/3) * x / size
    r = (-1/3) * x / size + (math.sqrt(3)/3) * y / size
    cx, cz = q, r; cy = -cx - cz
    rx, ry, rz = round(cx), round(cy), round(cz)
    dx, dy, dz = abs(rx - cx), abs(ry - cy), abs(rz - cz)
    if dx > dy and dx > dz:   rx = -ry - rz
    elif dy > dz:             ry = -rx - rz
    else:                     rz = -rx - ry
    return int(rx), int(rz)

def hex_polygon(q, r, size):
    cx = size * 1.5 * q
    cy = size * math.sqrt(3) * (r + q/2)
    return [((cy + size * math.sin(math.radians(60*i))) / ky,
              cx + size * math.cos(math.radians(60*i))) for i in range(6)]

# Accumulate sample counts per hex cell.
counts = Counter()
for lon, lat, n in weighted_points:
    counts[point_to_hex(lon, lat, HEX_SIZE_DEG)] += n

max_n = max(counts.values()) if counts else 1

# Five-step viridis ramp.
VIRIDIS = ["#440154", "#3b528b", "#21908d", "#5dc863", "#fde725"]
N_BINS  = len(VIRIDIS)

def bucket_colour(n):
    # Log-scaled bucketing because aDNA sample counts per cell are
    # heavily right-skewed.
    t = math.log(n) / math.log(max_n) if max_n > 1 else 1.0
    idx = min(N_BINS - 1, int(t * N_BINS))
    return VIRIDIS[idx]

# Back out legend bin edges from the log bucketing.
legend_items = []
if max_n > 1:
    for i in range(N_BINS):
        lo_t = i       / N_BINS
        hi_t = (i + 1) / N_BINS
        lo = max(1, int(math.ceil(math.exp(lo_t * math.log(max_n)))))
        hi = max(lo, int(math.floor(math.exp(hi_t * math.log(max_n)))))
        legend_items.append((VIRIDIS[i], f"{lo}" if lo == hi else f"{lo}–{hi}"))
else:
    legend_items.append((VIRIDIS[-1], "1"))

print(f"✓ {len(counts)} non-empty hex cells "
      f"(max samples per cell: {max_n:,}; total: {sum(counts.values()):,})")

centre = [df["latitude"].mean(), df["longitude"].mean()]
m2 = folium.Map(location=centre, zoom_start=3, tiles="OpenStreetMap")
folium.TileLayer(
    "https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    name="OSM Humanitarian",
    attr="&copy; OpenStreetMap contributors, Tiles style by HOT",
).add_to(m2)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Terrain",
    attr="Tiles &copy; Esri",
    max_zoom=13,
).add_to(m2)
Fullscreen(position="topleft").add_to(m2)

hex_layer = FeatureGroup(name="Hex density (samples per cell)")
for (q, r), n in counts.items():
    folium.Polygon(
        locations=hex_polygon(q, r, HEX_SIZE_DEG),
        color="#333", weight=0.6,
        fill=True, fill_color=bucket_colour(n), fill_opacity=0.78,
        tooltip=f"<b>{n:,}</b> aDNA sample{'s' if n != 1 else ''} in this cell",
    ).add_to(hex_layer)
hex_layer.add_to(m2)

# Legend in the bottom-right corner
legend_html = [
    '<div style="position:absolute;bottom:20px;right:20px;z-index:9999;',
    'background:rgba(255,255,255,0.92);padding:8px 12px;border-radius:4px;',
    'font:12px/1.5 system-ui,sans-serif;box-shadow:0 1px 4px rgba(0,0,0,0.2)">',
    '<b>Samples per hex</b> <small>(log)</small><br>',
]
# High bin on top
for colour, label in reversed(legend_items):
    legend_html.append(
        f'<span style="display:inline-block;width:14px;height:14px;'
        f'background:{colour};border:1px solid #333;margin-right:6px;'
        f'vertical-align:middle"></span>{label}<br>'
    )
legend_html.append("</div>")

class _Legend(MacroElement):
    _template = Template(
        "{% macro html(this, kwargs) %}"
        + "".join(legend_html)
        + "{% endmacro %}"
    )
m2.get_root().add_child(_Legend())

folium.LayerControl(collapsed=True).add_to(m2)

m2.fit_bounds([[df["latitude"].min(), df["longitude"].min()],
               [df["latitude"].max(), df["longitude"].max()]],
              padding=(20, 20))
m2

## Step 4 — Exploring the data

The cells below are a free playground — rank sites by sample
count, or compute per-country aggregates. Remember: one row of
`df` is one *(site, country, coordinate)* tuple, with a
`numSamples` column holding the number of aDNA samples aggregated
there.


In [ ]:
assert 'df' in dir(), "⚠ Please run the data-loading cell first."

# Top 15 sites by sample count — the "heavily-sampled-sites" list.
top = (df.sort_values("numSamples", ascending=False)
         .head(15)
         [["site", "country", "numSamples"]]
         .reset_index(drop=True))
print(f"Top {len(top)} sites by aDNA sample count:")
top

In [ ]:
assert 'df' in dir(), "⚠ Please run the data-loading cell first."

# Per-country summary: site count and total sample count.
summary = (
    df.groupby("country")
      .agg(
          n_sites=("site", "nunique"),
          n_samples=("numSamples", "sum"),
      )
      .sort_values("n_samples", ascending=False)
)
summary

---

## References

- **Straten, M. thor, Strohm, S., Thiery, F. & Renz, M.** (2025). Data-Driven
  Community Standards for Interdisciplinary Heterogeneous Information
  Networks. *E-Science-Tage 2025.* Heidelberg: heiBOOKS.
  doi: [10.11588/heibooks.1652.c23914](https://doi.org/10.11588/heibooks.1652.c23914)
- **Schmid, C. et al.** (2024). Poseidon — A framework for archaeogenetic
  human genotype data management. *eLife* 13.
  doi: [10.7554/eLife.98317.1](https://doi.org/10.7554/eLife.98317.1)
- **archaeonatural-cloud/poseidon2lod** — RDF generation scripts.
  [github.com/archaeonatural-cloud/poseidon2lod](https://github.com/archaeonatural-cloud/poseidon2lod)

*This notebook is part of the Open Educational Resources of
[NFDI4Objects](https://www.nfdi4objects.net/).*
